# Project 3: Eliminating Child Care Deserts in New York State through Optimization

In [1]:
#!pip install pulp

In [2]:
import numpy as numpy
import pandas as pd

### Data Cleaning

In [3]:
population = pd.read_csv("https://raw.githubusercontent.com/hayooniverse/IEORE4004-Optimization-Models-and-Methods/refs/heads/main/Project%203/data/population.csv")
employment = pd.read_csv("https://raw.githubusercontent.com/hayooniverse/IEORE4004-Optimization-Models-and-Methods/refs/heads/main/Project%203/data/employment_rate.csv")
income = pd.read_csv("https://raw.githubusercontent.com/hayooniverse/IEORE4004-Optimization-Models-and-Methods/refs/heads/main/Project%203/data/avg_individual_income.csv")
facilities = pd.read_csv("https://raw.githubusercontent.com/hayooniverse/IEORE4004-Optimization-Models-and-Methods/refs/heads/main/Project%203/data/child_care_regulated.csv")
locations = pd.read_csv("https://raw.githubusercontent.com/hayooniverse/IEORE4004-Optimization-Models-and-Methods/refs/heads/main/Project%203/data/potential_locations.csv")

In [4]:
income.rename(columns={
    "ZIP code" : "zipcode"
}, inplace=True)

#### See Data

In [5]:
population.head()

,zipcode,Total,-5,5-9,10-14,15-19,20-24,25-29,30-34,35-39,40-44,45-49,50-54,55-59,60-64,65-69,70-74,75-79,80-84,85+
0,6390,53,0,1,5,0,6,0,9,18,0,12,2,0,0,0,0,0,0,0
1,10001,27004,744,784,942,1035,2296,3806,3588,2524,1702,1903,1704,1225,1323,933,815,616,488,576
2,10002,76518,2142,3046,3198,2652,4528,6988,6278,5157,4962,4822,4410,6106,4548,4815,4748,2531,2793,2794
3,10003,53877,1440,1034,953,7013,6344,7100,6427,3221,2907,1988,2698,2350,2274,2793,1854,1646,779,1056
4,10004,4579,433,182,161,108,109,601,724,490,241,313,549,279,199,173,2,15,0,0


In [6]:
employment.head()

,zipcode,employment rate
0,10001,0.595097
1,10002,0.520662
2,10003,0.497244
3,10004,0.506661
4,10005,0.665833


In [7]:
income.head()

,zipcode,average income
0,10001,102878.033603
1,10002,59604.041165
2,10003,114273.049645
3,10004,132004.310345
4,10005,121437.713311


In [8]:
facilities.head()

,facility_id,program_type,facility_status,facility_name,city,zip_code,school_district_name,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,children_capacity,total_capacity,latitude,longitude
0,2416,FDC,Registration,"Bohrer, Barbara",Clinton,13323,Clinton,0,0,0,2,6,8,NaN,NaN
1,5555,FDC,Registration,"Matey, Sally",Jamestown,14701,Jamestown,0,0,0,2,6,8,NaN,NaN
2,9066,FDC,Registration,"Copeland, Denise",Wappingers Falls,12590,Wappingers,0,0,0,2,6,8,NaN,NaN
3,40163,DCC,License,"Head Start of Rockland, Inc.",Nyack,10960,Nyack,0,10,110,0,0,120,41.089425,-73.920413
4,41016,SACC,Registration,"School's Out, Inc.",Glenmont,12077,Bethlehem,0,0,0,75,0,75,42.607043,-73.788606


In [9]:
locations.head()

,zipcode,latitude,longitude
0,501,40.816376,-73.040796
1,501,40.817455,-73.044202
2,501,40.813873,-73.042182
3,501,40.814152,-73.037265
4,501,40.824673,-73.047431


#### Defining Demographic Information
* r_a: the rate of employment of area a
* e_a: the average income level of area a
* p_t_a : the total number of children ages 2 weeks - 12 years in area a
* p_u_a : the total number of children under 5 (babies and toddlers) in area a


In [10]:
# make a copy of employment rate variable
df = employment.copy()
# merge with income on zipcode 
df = df.merge(income, on="zipcode")
# create new column p_t_a: under 5 years old + 5-9 years old + 10-14 years
df["p_t_a"] = population["-5"] + population["5-9"] + population["10-14"]
# create new column p_u_a: under 5 
df["p_u_a"] = population["-5"]
df.rename(columns={"employment rate":"r_a","average income":"e_a"}, inplace=True)
df

,zipcode,r_a,e_a,p_t_a,p_u_a
0,10001,0.595097,102878.033603,6,0
1,10002,0.520662,59604.041165,2470,744
2,10003,0.497244,114273.049645,8386,2142
3,10004,0.506661,132004.310345,3427,1440
4,10005,0.665833,121437.713311,776,433
...,...,...,...,...,...
1370,14767,0.322296,54623.287671,42,0
1371,14770,0.446676,55523.255814,42,0
1372,14772,0.410719,57164.634146,1030,375
1373,14805,0.679739,59375.000000,188,20


Flagging high-demand areas (defined as regions where at least 60% of parents are employed or the average income is $60,000 or less per year)

In [11]:
df["high_demand"] = (df["r_a"] >= 0.60) | (df["e_a"] <= 60000)
df

,zipcode,r_a,e_a,p_t_a,p_u_a,high_demand
0,10001,0.595097,102878.033603,6,0,False
1,10002,0.520662,59604.041165,2470,744,True
2,10003,0.497244,114273.049645,8386,2142,False
3,10004,0.506661,132004.310345,3427,1440,False
4,10005,0.665833,121437.713311,776,433,True
...,...,...,...,...,...,...
1370,14767,0.322296,54623.287671,42,0,True
1371,14770,0.446676,55523.255814,42,0,True
1372,14772,0.410719,57164.634146,1030,375,True
1373,14805,0.679739,59375.000000,188,20,True


#### Defining Operational Information
- n_t_a_j : the total number of existing slots in facility j in area a for all age ranges
- n_u_a_j : the number of existing slots in facility j in area a for child under 5

Assumption: Here, we assumed that infants - preschool accounts for the number of children under 5

In [12]:
# adjust variable name
facilities.rename(columns={"zip_code":"zipcode","total_capacity": "n_t_a_j"}, inplace=True)
# cap for under 5 : infant + toddler + preschool
facilities["n_u_a_j"] = (facilities["infant_capacity"] + facilities["toddler_capacity"] + facilities["preschool_capacity"])
facilities

,facility_id,program_type,facility_status,facility_name,city,zipcode,school_district_name,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,children_capacity,n_t_a_j,latitude,longitude,n_u_a_j
0,2416,FDC,Registration,"Bohrer, Barbara",Clinton,13323,Clinton,0,0,0,2,6,8,NaN,NaN,0
1,5555,FDC,Registration,"Matey, Sally",Jamestown,14701,Jamestown,0,0,0,2,6,8,NaN,NaN,0
2,9066,FDC,Registration,"Copeland, Denise",Wappingers Falls,12590,Wappingers,0,0,0,2,6,8,NaN,NaN,0
3,40163,DCC,License,"Head Start of Rockland, Inc.",Nyack,10960,Nyack,0,10,110,0,0,120,41.089425,-73.920413,120
4,41016,SACC,Registration,"School's Out, Inc.",Glenmont,12077,Bethlehem,0,0,0,75,0,75,42.607043,-73.788606,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15599,892735,GFDC,License,LITTLE LILIES GROUP FAMILY DAYCARE LLC.,Bronx,10462,Bronx 11,0,0,0,4,12,16,40.854317,-73.864996,0
15600,897263,GFDC,License,"Cummings, Darlene",Brooklyn,11205,Brooklyn 13,0,0,0,0,10,10,40.696940,-73.977121,0
15601,901966,GFDC,License,"Pascal Genao, Angela",Yonkers,10705,Yonkers,0,0,0,4,12,16,40.910754,-73.893528,0
15602,892455,GFDC,License,"Warnakulasuriya, Sajeeka",Staten Island,10303,Richmond 31,0,0,0,4,12,16,40.628144,-74.156228,0


Now we group the dataset by the zipcodes, so n_t_a_j becomes n_t_a, and n_u_a_j becomes n_u_a.

In [13]:
temp_facilities = facilities.groupby("zipcode").agg({"n_t_a_j": "sum","n_u_a_j": "sum"}).reset_index()
temp_facilities.rename(columns={"n_t_a_j":"n_t_a","n_u_a_j":"n_u_a"}, inplace=True)
temp_facilities

,zipcode,n_t_a,n_u_a
0,10001,609,0
1,10002,4729,18
2,10003,1995,0
3,10004,263,0
4,10005,39,0
...,...,...,...
1599,148799621,8,0
1600,149011149,16,0
1601,149012216,82,82
1602,149042203,16,0


In [14]:
df = df.merge(temp_facilities, on="zipcode", how="left")
df["n_t_a"] = df["n_t_a"].fillna(0)
df["n_u_a"] = df["n_u_a"].fillna(0)
df

,zipcode,r_a,e_a,p_t_a,p_u_a,high_demand,n_t_a,n_u_a
0,10001,0.595097,102878.033603,6,0,False,609.0,0.0
1,10002,0.520662,59604.041165,2470,744,True,4729.0,18.0
2,10003,0.497244,114273.049645,8386,2142,False,1995.0,0.0
3,10004,0.506661,132004.310345,3427,1440,False,263.0,0.0
4,10005,0.665833,121437.713311,776,433,True,39.0,0.0
...,...,...,...,...,...,...,...,...
1370,14767,0.322296,54623.287671,42,0,True,16.0,0.0
1371,14770,0.446676,55523.255814,42,0,True,70.0,15.0
1372,14772,0.410719,57164.634146,1030,375,True,108.0,32.0
1373,14805,0.679739,59375.000000,188,20,True,8.0,0.0


#### 1. Child Care Desert Classification
- In high-demand areas: if the number of available slots is less than or equal to half the population of children aged two weeks to 12 years
- In normal-demand areas: if the available slots are less than or equal to one-third of the population of children within the same age range

In [15]:
df["is_desert"] = df.apply(lambda row: row["n_t_a"] <= 0.5 * row["p_t_a"] if row["high_demand"] else row["n_t_a"] <= (1/3) * row["p_t_a"],axis=1)
df

,zipcode,r_a,e_a,p_t_a,p_u_a,high_demand,n_t_a,n_u_a,is_desert
0,10001,0.595097,102878.033603,6,0,False,609.0,0.0,False
1,10002,0.520662,59604.041165,2470,744,True,4729.0,18.0,False
2,10003,0.497244,114273.049645,8386,2142,False,1995.0,0.0,True
3,10004,0.506661,132004.310345,3427,1440,False,263.0,0.0,True
4,10005,0.665833,121437.713311,776,433,True,39.0,0.0,True
...,...,...,...,...,...,...,...,...,...
1370,14767,0.322296,54623.287671,42,0,True,16.0,0.0,True
1371,14770,0.446676,55523.255814,42,0,True,70.0,15.0,False
1372,14772,0.410719,57164.634146,1030,375,True,108.0,32.0,True
1373,14805,0.679739,59375.000000,188,20,True,8.0,0.0,True


#### 2. Age 0-5 Child Care Desert Classification
* Children under the age of 5 must have sufficient access to care
* The number of available slots for children in this age group must be at least two-thirds of the population of children aged 0-5



In [16]:
# n_u_a is the number of existing slots for child under 5
# p_u_a is the number of children under 5 
df["is_under5_desert"] = df["n_u_a"] < (2/3) * df["p_u_a"]
df

,zipcode,r_a,e_a,p_t_a,p_u_a,high_demand,n_t_a,n_u_a,is_desert,is_under5_desert
0,10001,0.595097,102878.033603,6,0,False,609.0,0.0,False,False
1,10002,0.520662,59604.041165,2470,744,True,4729.0,18.0,False,True
2,10003,0.497244,114273.049645,8386,2142,False,1995.0,0.0,True,True
3,10004,0.506661,132004.310345,3427,1440,False,263.0,0.0,True,True
4,10005,0.665833,121437.713311,776,433,True,39.0,0.0,True,True
...,...,...,...,...,...,...,...,...,...,...
1370,14767,0.322296,54623.287671,42,0,True,16.0,0.0,True,False
1371,14770,0.446676,55523.255814,42,0,True,70.0,15.0,False,False
1372,14772,0.410719,57164.634146,1030,375,True,108.0,32.0,True,True
1373,14805,0.679739,59375.000000,188,20,True,8.0,0.0,True,True


## Problem A: Budgeting
Determining the minimum amount of funding (in total) needed to meet their target for each area, categorized by zip code

#### Further Clean Facilities Data

In [17]:
facilities = pd.read_csv("https://raw.githubusercontent.com/hayooniverse/IEORE4004-Optimization-Models-and-Methods/refs/heads/main/Project%203/data/child_care_regulated.csv")
facilities["under_5_capacity"] = (facilities["infant_capacity"].fillna(0) + facilities["toddler_capacity"].fillna(0) + facilities["preschool_capacity"].fillna(0))
facilities

,facility_id,program_type,facility_status,facility_name,city,zip_code,school_district_name,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,children_capacity,total_capacity,latitude,longitude,under_5_capacity
0,2416,FDC,Registration,"Bohrer, Barbara",Clinton,13323,Clinton,0,0,0,2,6,8,NaN,NaN,0
1,5555,FDC,Registration,"Matey, Sally",Jamestown,14701,Jamestown,0,0,0,2,6,8,NaN,NaN,0
2,9066,FDC,Registration,"Copeland, Denise",Wappingers Falls,12590,Wappingers,0,0,0,2,6,8,NaN,NaN,0
3,40163,DCC,License,"Head Start of Rockland, Inc.",Nyack,10960,Nyack,0,10,110,0,0,120,41.089425,-73.920413,120
4,41016,SACC,Registration,"School's Out, Inc.",Glenmont,12077,Bethlehem,0,0,0,75,0,75,42.607043,-73.788606,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15599,892735,GFDC,License,LITTLE LILIES GROUP FAMILY DAYCARE LLC.,Bronx,10462,Bronx 11,0,0,0,4,12,16,40.854317,-73.864996,0
15600,897263,GFDC,License,"Cummings, Darlene",Brooklyn,11205,Brooklyn 13,0,0,0,0,10,10,40.696940,-73.977121,0
15601,901966,GFDC,License,"Pascal Genao, Angela",Yonkers,10705,Yonkers,0,0,0,4,12,16,40.910754,-73.893528,0
15602,892455,GFDC,License,"Warnakulasuriya, Sajeeka",Staten Island,10303,Richmond 31,0,0,0,4,12,16,40.628144,-74.156228,0


In [18]:
facilities_dict = {}
for _,row in facilities.iterrows():
    zipcode = row["zip_code"]
    facility_info = (int(row["facility_id"]), int(row["total_capacity"]), int(row["under_5_capacity"]))
    if zipcode not in facilities_dict:
        facilities_dict[zipcode] = []
    facilities_dict[zipcode].append(facility_info)

### Objective
The objective of Problem A is to determine the minimum amount of funding (in total) needed to meet the NYS government's target for each area, categorized by zip code.

The government plans to either:
1. build new child care facilities
2. expand exisiting ones

For this problem, we decided to use the PuLP library.
We referred to this page to learn more about the PuLP library.
https://www.geeksforgeeks.org/python-linear-programming-in-pulp/

#### Create a LP Minimization Problem

In [19]:
import pulp as p
# create minimization problem
model = p.LpProblem("Child_Care_Desert_Elimination", p.LpMinimize)

#### Define Costs
1. Small: Facility Size = 100, # of Slots (Ages 0-5) = 50, Cost of New Facility = 65000
2. Medium: Facility Size = 200, # of Slots (Ages 0-5) = 100, Cost of New Facility = 95000
3. Large: Facility Size = 400, # of Slots (Ages 0-5) = 200, Cost of New Facility = 115,000

In [20]:
new_facility_total_slots = {1: 100, 2: 200, 3: 400}
new_facility_under5_slots = {1: 50, 2: 100, 3: 200}
new_facility_cost = {1: 65000, 2: 95000, 3: 115000}

#### Create Decision Variables
- y_a_i : the number of new facility of type i to build in area a
- x_a_j : the ratio of extended slots over the current slots in facility j in area a
- m_t_a_j : the total number of extended slots for children in facility j in area a
- m_u_a_j : the number of extended slots for children ages 0-5 in facility j in area a
- b_a_j: whether the facility j in area a is expanded (=1 if expanded and 0 otherwise)

In [21]:
# load zipcode to list 
zipcodes = df["zipcode"].tolist()
# load facilities to list 
facility_id = facilities['facility_id']


In [22]:
# create y_a_i
y = {(a, i): p.LpVariable(f"y_{a}_{i}", lowBound=0, cat=p.LpInteger)
     for a in zipcodes for i in [1, 2, 3]}

In [23]:
y

{(10001, 1): y_10001_1,
 (10001, 2): y_10001_2,
 (10001, 3): y_10001_3,
 (10002, 1): y_10002_1,
 (10002, 2): y_10002_2,
 (10002, 3): y_10002_3,
 (10003, 1): y_10003_1,
 (10003, 2): y_10003_2,
 (10003, 3): y_10003_3,
 (10004, 1): y_10004_1,
 (10004, 2): y_10004_2,
 (10004, 3): y_10004_3,
 (10005, 1): y_10005_1,
 (10005, 2): y_10005_2,
 (10005, 3): y_10005_3,
 (10006, 1): y_10006_1,
 (10006, 2): y_10006_2,
 (10006, 3): y_10006_3,
 (10007, 1): y_10007_1,
 (10007, 2): y_10007_2,
 (10007, 3): y_10007_3,
 (10009, 1): y_10009_1,
 (10009, 2): y_10009_2,
 (10009, 3): y_10009_3,
 (10010, 1): y_10010_1,
 (10010, 2): y_10010_2,
 (10010, 3): y_10010_3,
 (10011, 1): y_10011_1,
 (10011, 2): y_10011_2,
 (10011, 3): y_10011_3,
 (10012, 1): y_10012_1,
 (10012, 2): y_10012_2,
 (10012, 3): y_10012_3,
 (10013, 1): y_10013_1,
 (10013, 2): y_10013_2,
 (10013, 3): y_10013_3,
 (10014, 1): y_10014_1,
 (10014, 2): y_10014_2,
 (10014, 3): y_10014_3,
 (10016, 1): y_10016_1,
 (10016, 2): y_10016_2,
 (10016, 3): y_1

In [24]:
# create x_a_j
x = {(a, j): p.LpVariable(f"x_{a}_{j}", lowBound=0, upBound =0.2, cat=p.LpContinuous)
     for a in zipcodes for j in facility_id}

In [25]:
# create m_t_a_j
m_t = {(a, j): p.LpVariable(f"m_t_{a}_{j}", lowBound=0, cat=p.LpInteger)
     for a in zipcodes for j in facility_id}

In [26]:
m_u = {(a, j): p.LpVariable(f"m_u_{a}_{j}", lowBound=0, cat=p.LpInteger)
     for a in zipcodes for j in facility_id}

#### Create Objective Function

In [27]:
# Expansion based on size ( less than 10%, 10%-15%, 15%-20%) as given in page 3 
expansion_cost = p.lpSum(
    (20000 + (200 if x[(a, j)] <= 0.1 else 400 if x[(a, j)] <= 0.15 else 1000) * nt) * x[(a, j)]
    # additional cost per slot for children under 5
    + 100 * m_u[(a, j)]
    for a in zipcodes for (j, nt, _) in facilities.get(a, [])
)

In [28]:
# expansion cost + new facility cost (determined by size - small:1, medium:2, large:3)
model += expansion_cost + p.lpSum(y[(a, i)] * new_facility_cost[i] for a in zipcodes for i in [1, 2, 3])


#### Create Constraints

In [29]:
for idx, row in df.iterrows():
    a = row["zipcode"]
    existing_total = row["n_t_a"]
    existing_under5 = row["n_u_a"]
    p_t = row["p_t_a"]
    p_u = row["p_u_a"]
    threshold = 0.5 if row["high_demand"] else (1/3)

    # Total slot coverage
    model += (
        existing_total +
        p.lpSum(m_t[(a, j)] for (j, _, _) in facilities.get(a, [])) +
        p.lpSum(new_facility_total_slots[i] * y[(a, i)] for i in [1, 2, 3])
        >= threshold * p_t
    )

    # Under-5 slot coverage
    model += (
        existing_under5 +
        p.lpSum(m_u[(a, j)] for (j, _, _) in facilities.get(a, [])) +
        p.lpSum(new_facility_under5_slots[i] * y[(a, i)] for i in [1, 2, 3])
        >= (2/3) * p_u
    )


In [30]:
# Expansion constraints
for a in zipcodes:
    for (j, nt, _) in facilities.get(a, []):
        # expansion rate constraint as given: expansion ratio * current 
        model += m_t[(a, j)] == x[(a, j)] * nt
        # under-5 slots can't exceed the total num. of newly added 
        model += m_u[(a, j)] <= m_t[(a, j)]
        # maximum 500 slots per facility
        if nt >= 500:
            model += x[(a, j)] == 0 # force 0 when facility is already 500 

In [31]:
# Budget Constraint
budget_limit = 100_000_000  # $100 million
model += (
    expansion_cost +
    p.lpSum(y[(a, i)] * new_facility_cost[i] for a in zipcodes for i in [1, 2, 3])
    <= budget_limit
), "Budget_Constraint"

In [32]:
# Solve Model 
model.solve(p.PULP_CBC_CMD(msg=True)) 
print(f"Status: {p.LpStatus[model.status]}")
print(f"Objective Cost: ${p.value(model.objective):,.2f}")

# Output new facilities to build
build_rows = []
for (a, i), var in y.items():
    if var.varValue and var.varValue > 0:
        build_rows.append({
            "zipcode": a,
            "facility_type": i,
            "num_to_build": int(var.varValue),
            "slots_added": new_facility_total_slots[i] * int(var.varValue),
            "under5_slots_added": new_facility_under5_slots[i] * int(var.varValue),
        })

df_builds = pd.DataFrame(build_rows)

print("New Facilities to Build:")
display(df_builds)

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/pp/n1knqjg57ds10zbbb7v7yvfw0000gn/T/f5d9f0a04a8f4cfa8236b03582ee3d2f-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/pp/n1knqjg57ds10zbbb7v7yvfw0000gn/T/f5d9f0a04a8f4cfa8236b03582ee3d2f-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 2756 COLUMNS
At line 27507 RHS
At line 30259 BOUNDS
At line 34385 ENDATA
Problem MODEL has 2751 rows, 4125 columns and 12375 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Problem is infeasible - 0.01 seconds
Option for printingOptions changed from normal to all
Total time (CPU seconds):       0.03   (Wallclock seconds):       0.05

Status: Infeasible
Objective Cost: $111,715,504.32
New Facilities to Build:


,zipcode,facility_type,num_to_build,slots_added,under5_slots_added
0,10026,3,13,5200,2600
1,10027,3,14,5600,2800
2,10501,3,17,6800,3400
3,10504,3,15,6000,3000
4,10505,3,22,8800,4400
5,10509,3,18,7200,3600
6,10510,3,20,8000,4000
7,10511,3,18,7200,3600
8,10518,3,19,7600,3800
9,10524,3,14,5600,2800


In [33]:
expand_rows = []
for (a, j), var in x.items():
    if var.varValue and var.varValue > 0:
        original_capacity = next(nt for (fid, nt, _) in facilities[a] if fid == j)
        under5_capacity = next(nu for (fid, _, nu) in facilities[a] if fid == j)
        percent_increase = round(100 * var.varValue, 2)
        expand_rows.append({
            "zipcode": a,
            "facility_id": j,
            "expand_%": percent_increase,
            "added_slots": round(var.varValue * original_capacity),
            "under5_capacity_existing": under5_capacity,
        })

df_expansions = pd.DataFrame(expand_rows)
print("Facilities to Expand:")
display(df_expansions)

Facilities to Expand:


""


Our model suggests that for New York to eliminate all child care deserts while minimizing the cost, new facilities should be built rather than expanding existing ones.